### Setup

In [ ]:
import json
import shutil

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image as ShowImage
from IPython.display import display
from PIL import Image, ImageOps

from dogbreeds.paths import DATA_DIR, OUTPUTS_DIR, PROJECT_ROOT, RAW_DIR

RUNS_DIR = OUTPUTS_DIR / "runs"
FRESH_DIR = DATA_DIR / "fresh"
FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"
FIGURES_DIR.mkdir(exist_ok=True, parents=True)

# Display name -> fun folder. Theese are the official runs (best epoch chosen by val loss).
RUNS = {
    "v1 baseline": "v1_baseline_v2",
    "EfficientNet-B0": "efficientnet_b0_v2",
    "ConvNeXt-Tiny": "convnext_tiny_v2",
}
FINAL_RUN = "convnext_tiny_v2"
FIRST_CONVNEXT_RUN = "convnext_tiny"  # the first run, best epoch chosen by val accuracy
THRESHOLD = 0.8

# One fixed color per model
MODEL_COLORS = {"v1 baseline": "#2a78d6", "EfficientNet-B0": "#eb6834", "ConvNeXt-Tiny": "#1baf7a"}
GRID_COLOR = "#e5e5e5"
MUTED_TEXT = "#52514e"


def read_json(path):
    with open(path) as f:
        return json.load(f)


def fresh_accuracy(run):
    """Share of fresh photos whose predicted breed matches the folder they are in."""
    fresh = pd.read_csv(RUNS_DIR / run / "predictions_fresh.csv")
    return (fresh["predicted_breed"] == fresh["true_breed"]).mean()

### Compairson table

In [ ]:
rows = []
for name, run in RUNS.items():
    val = read_json(RUNS_DIR / run / "metrics_val.json")
    test = read_json(RUNS_DIR / run / "metrics_test.json")
    rows.append(
        {
            "model": name,
            "val accuracy %": round(val["accuracy"] * 100, 1),
            "test accuracy %": round(test["accuracy"] * 100, 1),
            "test macro F1 %": round(test["macro_f1"] * 100, 1),
            "fresh accuracy % (60 photos)": round(fresh_accuracy(run) * 100, 1),
        }
    )
comparison = pd.DataFrame(rows).set_index("model")
display(comparison)

### Learning curves

In [ ]:
# Validation loss and accuracy per epoch. The dashed line is the first ConvNeXt run
# (lr 3e-4, best epoch chosen by accuracy) for comparison.
fig, (loss_ax, acc_ax) = plt.subplots(1, 2, figsize=(11, 4))
for name, run in RUNS.items():
    history = pd.read_csv(RUNS_DIR / run / "history.csv")
    color = MODEL_COLORS[name]
    loss_ax.plot(history["epoch"], history["val_loss"], color=color, linewidth=2, label=name)
    acc_ax.plot(history["epoch"], history["val_acc"], color=color, linewidth=2, label=name)

first = pd.read_csv(RUNS_DIR / FIRST_CONVNEXT_RUN / "history.csv")
first_style = {"color": MODEL_COLORS["ConvNeXt-Tiny"], "linestyle": "--"}
first_label = "ConvNeXt-Tiny, first run"
loss_ax.plot(first["epoch"], first["val_loss"], label=first_label, **first_style)
acc_ax.plot(first["epoch"], first["val_acc"], label=first_label, **first_style)

loss_ax.set_title("Validation loss")
acc_ax.set_title("Validation accuracy")
for ax in (loss_ax, acc_ax):
    ax.set_xlabel("Epoch")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color=GRID_COLOR)
    ax.set_axisbelow(True)
acc_ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "learning_curves.png", dpi=150)
plt.show()

### Final model's test confusion matrix

In [ ]:
confusion_png = RUNS_DIR / FINAL_RUN / "confusion_test.png"
shutil.copy(confusion_png, FIGURES_DIR / "confusion_test.png")
display(ShowImage(filename=str(confusion_png)))

### Helper functions for showing photos

In [ ]:
def show_images(paths, captions, columns, title, save_name=None):
    """Show photos in a grid with a caption under each; optionally save the figure."""
    rows = (len(paths) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 3, rows * 3.3), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for i, path in enumerate(paths):
        with Image.open(path) as img:
            # Phone photos store rotation in EXIF; apply it so they display upright.
            axes.flat[i].imshow(ImageOps.exif_transpose(img).convert("RGB"))
        axes.flat[i].set_title(captions[i], fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    if save_name:
        fig.savefig(FIGURES_DIR / save_name, dpi=80, pil_kwargs={"quality": 80})
    plt.show()


def mistake_caption(row):
    """Two-line caption: the true breed, then the prediction and its confidence."""
    predicted = f"{row['predicted_breed']} ({row['confidence']:.2f})"
    return f"true: {row['true_breed']}\npredicted: {predicted}"

### Final model's mistakes

In [ ]:
test = pd.read_csv(RUNS_DIR / FINAL_RUN / "predictions_test.csv")
test_wrong = test[test["predicted_breed"] != test["true_breed"]]
paths = [RAW_DIR / image for image in test_wrong["image"]]
captions = [mistake_caption(row) for _, row in test_wrong.iterrows()]
show_images(paths, captions, 5, "ConvNeXt-Tiny mistakes on the test split", "test_mistakes.jpg")

# Fresh photos are shown but not saved
fresh = pd.read_csv(RUNS_DIR / FINAL_RUN / "predictions_fresh.csv")
fresh_wrong = fresh[fresh["predicted_breed"] != fresh["true_breed"]]
paths = [FRESH_DIR / path for path in fresh_wrong["path"]]
captions = [mistake_caption(row) for _, row in fresh_wrong.iterrows()]
show_images(paths, captions, 5, "ConvNeXt-Tiny mistakes on the fresh photos")

### Threshold trade-off

In [ ]:
# Known dogs = 12 breeds (test split + fresh photos); unknown = 540 dogs of other breeds.
# Below the threshold the model answers "not sure" instead of a breed.
known_confidence = pd.concat([test["confidence"], fresh["confidence"]])
unknown = pd.read_csv(RUNS_DIR / FINAL_RUN / "predictions_unknown.csv")

thresholds = [step / 100 for step in range(0, 100)]
known_rejected = [(known_confidence < t).mean() for t in thresholds]
unknown_rejected = [(unknown["confidence"] < t).mean() for t in thresholds]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    thresholds,
    unknown_rejected,
    color=MODEL_COLORS["v1 baseline"],
    linewidth=2,
    label="unknown breeds rejected (good)",
)
ax.plot(
    thresholds,
    known_rejected,
    color=MODEL_COLORS["EfficientNet-B0"],
    linewidth=2,
    label="our 12 breeds rejected (bad)",
)
ax.axvline(THRESHOLD, color=MUTED_TEXT, linestyle="--", linewidth=1.5)
ax.text(THRESHOLD - 0.01, 0.5, f"chosen: {THRESHOLD}", color=MUTED_TEXT, ha="right")
ax.set_title("ConvNeXt-Tiny: what a confidence threshold rejects")
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Share of images rejected")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(color=GRID_COLOR)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "threshold_tradeoff.png", dpi=150)
plt.show()

for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    known_share = (known_confidence < t).mean()
    unknown_share = (unknown["confidence"] < t).mean()
    print(f"threshold {t}: rejects {unknown_share:.1%} unknown, {known_share:.1%} of our breeds")

### Unknown Breeds 

In [ ]:
accepted = unknown[unknown["confidence"] >= THRESHOLD].copy()
accepted["unknown_breed"] = accepted["folder"].str.split("-", n=1).str[1]
slipped = accepted.groupby(["unknown_breed", "predicted_breed"]).size()
print(f"{len(accepted)} of {len(unknown)} unknown dogs accepted at threshold {THRESHOLD}")
display(slipped.sort_values(ascending=False).head(15).to_frame("images"))

Overall, these results show a massive improvement from the scratch CNN model to transfer learning. The scratch CNN had to learn everything about what a dog looks like from only roughly 1,500 training images, which is far too few. So it mostly learned simple cues like coat color and reached about 30%. The pretrained models had already learned general visual features (edges, textures, fur, ears and faces) from over a million ImageNet images, so they only had to adapt that knowledge to our 12 breeds, which took them to 96–98.5%. More time could have been put into refining the CNN model, but the main focus was transfer learning. 

A big surprise was the head-only ConvNeXt where in the first run, the best checkpoint was from epoch 2, before the full network had been fine-tuned. Because ConvNeXt was pretrained on ImageNet, which already contains these breeds, it reached 99% accuracy with only its final layer trained, so choosing the best epoch by accuracy picked a model that wasn't properly fine-tuned. I therefore switched to choosing the best epoch by validation loss so that all the models are as fine-tuned as possible. 

The look-alike pairs from notebook 01 (Basset vs. Bloodhound and Maltese vs. Shih-Tzu, along with Yorkshire Terrier) are still present and account for the majority of the real errors. For example, Shih-Tzu -> Maltese was the most common mistake for all three models on the test split. The fresh photos that I hand-picked from outside sources verified the validity of the models' accuracy in which ConvNeXt got 59 of 60 correct on photos it could not have seen before. 

Finally, since the model always picks one of the 12 breeds, a confidence threshold of 0.8 lets it answer "not sure" instead. This rejects about 90% of dogs from the 108 other Stanford breeds while turning away only a small share of photos of our 12 breeds.